In [1]:
import os
import requests
import streamlit as st
import json
import google.generativeai as genai
from dotenv import load_dotenv
import pandas as pd
from langchain_community.document_loaders import PyPDFLoader,JSONLoader,CSVLoader, WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_ollama import OllamaEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain.chains import create_retrieval_chain
from langchain.tools.retriever import create_retriever_tool
from langchain.agents import create_openai_tools_agent
from langchain_openai import OpenAIEmbeddings
from langchain.chains.combine_documents import create_stuff_documents_chain

USER_AGENT environment variable not set, consider setting it to identify your requests.


In [2]:
web_text=WebBaseLoader("https://uiic.co.in/downloadforms/downloads")

In [3]:
import json

# Read the file
with open('C:/Projects/bolt/Dataset for DS Case Study.json', 'r') as file:
    lines = file.readlines()
    documents = []
    for line in lines:
        try:
            documents.append(json.loads(line.strip()))
        except json.JSONDecodeError as e:
            print(f"Error decoding JSON: {e}")


In [4]:
df=pd.DataFrame(documents)
df=df.to_csv('C:/Projects/bolt/Dataset_for_DS_Case_Study.csv',index=False)

In [5]:
csv_documents=CSVLoader("C:/Projects/bolt/DS_Case_Study.csv")
csv_load=csv_documents.load()
print(csv_load[0].page_content)

reviewerID: A3F73SC1LY51OO
asin: B00002243X
reviewerName: Alan Montgomery
helpful: [4, 4]
reviewText: I needed a set of jumper cables for my new car and these had good reviews and were at a good price.  They have been used a few times already and do what they are supposed to - no complaints there.What I will say is that 12 feet really isn't an ideal length.  Sure, if you pull up front bumper to front bumper they are plenty long, but a lot of times you will be beside another car or can't get really close.  Because of this, I would recommend something a little longer than 12'.Great brand - get 16' version though.
overall: 5
summary: Work Well - Should Have Bought Longer Ones
unixReviewTime: 1313539200
reviewTime: 08 17, 2011


## Text Splitter (Chunks):-

In [6]:
text_split=RecursiveCharacterTextSplitter(separators="\n\n",chunk_size=5000,chunk_overlap=500)
text_chunk=text_split.split_documents(csv_load)
print(text_chunk[0].page_content)

reviewerID: A3F73SC1LY51OO
asin: B00002243X
reviewerName: Alan Montgomery
helpful: [4, 4]
reviewText: I needed a set of jumper cables for my new car and these had good reviews and were at a good price.  They have been used a few times already and do what they are supposed to - no complaints there.What I will say is that 12 feet really isn't an ideal length.  Sure, if you pull up front bumper to front bumper they are plenty long, but a lot of times you will be beside another car or can't get really close.  Because of this, I would recommend something a little longer than 12'.Great brand - get 16' version though.
overall: 5
summary: Work Well - Should Have Bought Longer Ones
unixReviewTime: 1313539200
reviewTime: 08 17, 2011


In [7]:
print(text_chunk[1].page_content)

reviewerID: A20S66SKYXULG2
asin: B00002243X
reviewerName: alphonse
helpful: [1, 1]
reviewText: These long cables work fine for my truck, but the quality seems a little on the shabby side. For the money I was not expecting 200 dollar snap-on jumper cables but these seem more like what you would see at a chinese knock off shop like harbor freight for 30 bucks.
overall: 4
summary: Okay long cables
unixReviewTime: 1315094400
reviewTime: 09 4, 2011


In [8]:
print(text_chunk[2].page_content)

reviewerID: A2I8LFSN2IS5EO
asin: B00002243X
reviewerName: Chris
helpful: [0, 0]
reviewText: Can't comment much on these since they have not yet been used (I will come back and update my review is I find any issues after first use) ... but they are built solid, nice tough big hard clamps and love having a long cable so I never have to move cars around or anything if needed. I bought these to have in my new truck since you always need cables ... but another reason is for when I tow my travel trailer and we run the batteries with no shore power they may die after a couple days ... if you are in need a quick small recharge they are the perfect length to pop my hood, run the cables to the back of the truck and hook up to the batteries that are on the tongue of my travel trailer ... running the truck for 30-45 minutes with this nice large gauge wire connected from my battery tot he trailer battery will give me a bit of a charge if ever in a pinch and I have no shore power, solar, or generato

In [9]:
print(text_chunk[3].page_content)

reviewerID: A3GT2EWQSO45ZG
asin: B00002243X
reviewerName: DeusEx
helpful: [19, 19]
reviewText: I absolutley love Amazon!!!  For the price of a set of cheap Booster/Jumper Cables in a brick and morter store, you can buy extra long and heavy duty jumpers!  First off, don't be the person that not only needs to ask a kind passer-by for a "jump" but also if they have jumper cables.  It's MUCH easier to get a jump start if you have your own cables.Next lets talk about sizing.  Having the longest cable possible is a major plus if your car is parked up against something like a pole or wall, or even parked on a one way street.  The "booster car" (the car w/o a dead battery) can pull in close enough to use the cables without having to manuver into some akward position.  Or better yet, you won't have to push your vehicle into a position to be jumped.  If your diving a normal sized car they can even pull in behind you to jump you!  Or if their vehicle is the shorter of the two, they could pull in 

## Embedding (Vector):-

In [10]:
text_embeddings=OllamaEmbeddings(model="gemma2:2b")

# Vectore Store Data Base

In [11]:
text_db=FAISS.from_documents(text_chunk,text_embeddings)
retriever=text_db.as_retriever()

In [12]:
retriever

VectorStoreRetriever(tags=['FAISS', 'OllamaEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x0000014659278D30>, search_kwargs={})

# Convert Retriever into Tools

In [13]:
retriever_tool=create_retriever_tool(retriever,"bolt document","search asny information asbout langsmith")

In [14]:
retriever_tool.name

'bolt document'

## Combine the all tools

In [76]:
tools=[retriever_tool]

In [77]:
tools

[Tool(name='bolt document', description='search asny information asbout langsmith', args_schema=<class 'langchain_core.tools.retriever.RetrieverInput'>, func=functools.partial(<function _get_relevant_documents at 0x0000014652AB6DD0>, retriever=VectorStoreRetriever(tags=['FAISS', 'OllamaEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x0000014659278D30>, search_kwargs={}), document_prompt=PromptTemplate(input_variables=['page_content'], input_types={}, partial_variables={}, template='{page_content}'), document_separator='\n\n'), coroutine=functools.partial(<function _aget_relevant_documents at 0x0000014652CBB250>, retriever=VectorStoreRetriever(tags=['FAISS', 'OllamaEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x0000014659278D30>, search_kwargs={}), document_prompt=PromptTemplate(input_variables=['page_content'], input_types={}, partial_variables={}, template='{page_content}'), document_separator='\n\n'))]

# Execute Tools and LLM with Agent Excutor
## Run all the tools with Agents and LLM model

In [78]:
from langchain_groq import ChatGroq
from dotenv import load_dotenv
load_dotenv()

True

In [79]:
groq_api_key=os.getenv("GROQ_API_KEY") 
llm=ChatGroq(groq_api_key=groq_api_key,model="Llama3-8b-8192")

In [80]:
from langchain_core.prompts import ChatPromptTemplate,MessagesPlaceholder

In [81]:
from langchain import hub

In [108]:
prompt = hub.pull("hwchase17/openai-functions-agent")

In [109]:
prompt.messages

[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='You are a helpful assistant'), additional_kwargs={}),
 MessagesPlaceholder(variable_name='chat_history', optional=True),
 HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, template='{input}'), additional_kwargs={}),
 MessagesPlaceholder(variable_name='agent_scratchpad')]

In [110]:
from langchain.prompts import PromptTemplate
prompt_tem=PromptTemplate(input_variable=["message"], template="Aske {message}")

In [111]:
prompt_new= ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are very powerful assistant, but don't know current events",
        ),
        ("user", "{input}"),
        MessagesPlaceholder(variable_name="agent_scratchpad"),
    ]
)

# Excute altogether 
## Agent

In [112]:
from langchain.agents import create_openai_tools_agent

In [113]:
agent=create_openai_tools_agent(llm=llm,tools=tools,prompt=prompt)

In [114]:
agent

RunnableAssign(mapper={
  agent_scratchpad: RunnableLambda(lambda x: format_to_openai_tool_messages(x['intermediate_steps']))
})
| ChatPromptTemplate(input_variables=['agent_scratchpad', 'input'], optional_variables=['chat_history'], input_types={'chat_history': list[typing.Annotated[typing.Union[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')], typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')], typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')], typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')], typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')], typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')], typing.Annotated[langchain_core.messages.ai.AIMessageChunk, Tag(tag='AIMessageChunk')], typing.Annotated[langchain_core.messages.human.HumanMessageChunk, Tag(tag='HumanMessageChunk')], typing.Annotated[langchain_core.messages

## Agent Executer to run the Agent

In [115]:
from langchain.agents import AgentExecutor

In [116]:
agent_executor=AgentExecutor(agent=agent,tools=tools)

In [117]:
agent_executor

AgentExecutor(verbose=False, agent=RunnableMultiActionAgent(runnable=RunnableAssign(mapper={
  agent_scratchpad: RunnableLambda(lambda x: format_to_openai_tool_messages(x['intermediate_steps']))
})
| ChatPromptTemplate(input_variables=['agent_scratchpad', 'input'], optional_variables=['chat_history'], input_types={'chat_history': list[typing.Annotated[typing.Union[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')], typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')], typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')], typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')], typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')], typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')], typing.Annotated[langchain_core.messages.ai.AIMessageChunk, Tag(tag='AIMessageChunk')], typing.Annotated[langchain_core.messages.human.HumanMessageChunk, Ta

In [118]:
query="which is the best review from reviewText"

In [119]:
agent_executor.invoke({"input":query})

{'input': 'which is the best review from reviewText',
 'output': 'Based on the reviewText, I would say that the best review is:\n\n"One of the best things I have bought to keep the inside of my car clean and looking new.  I recommend this for everyone."\n\nThis review is from reviewerID: A30XYS6AQ17DNK and has an overall rating of 5 out of 5 stars. It is a very positive and enthusiastic review, and the reviewer highly recommends the product.'}

In [134]:
query1="who is most reviewerName unhappy by seeing reviewText "

In [135]:
agent_executor.invoke({"input":query1})

{'input': 'who is most reviewerName unhappy by seeing reviewText ',
 'output': 'ACAR'}

In [127]:
query2="who is most happy reviewerName by seeing reviewText "

In [128]:
agent_executor.invoke({"input":query2})

{'input': 'who is most happy reviewerName by seeing reviewText ',
 'output': 'Based on the provided reviews, I would say that Gulfcoast is the most happy reviewer, as they gave the product a 5-star review and mentioned that it "works as described" and that they would "suggest to anyone looking for a product like this". They also mentioned that the installation was easy and took less than 5 minutes.'}